# Day 041 Solution — Natural Language → Pandas

get_df_schema, build_query_prompt, extract_code, run_pandas_code, ask_df. All data and functions defined inline.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import re
import ollama
import pandas as pd
import io


def get_df_schema(df) -> str:
    lines = [f"Shape: {df.shape[0]} rows x {df.shape[1]} columns"]
    lines.append("\nColumns and dtypes:")
    for col, dtype in df.dtypes.items():
        lines.append(f"  {col}: {dtype}")
    lines.append(f"\nSample (first 3 rows):\n{df.head(3).to_string(index=False)}")
    return "\n".join(lines)


def build_query_prompt(question: str, schema_str: str) -> str:
    return (
        "You are a Python data analyst. Write pandas code to answer the question.\n\n"
        "Requirements:\n"
        "- The DataFrame is already loaded as `df`. `pd` is also in scope.\n"
        "- Store the final answer in a variable named `result`.\n"
        "- Respond with ONLY a fenced Python code block, no explanation.\n\n"
        f"DataFrame schema:\n{schema_str}\n\n"
        f"Question: {question}"
    )


import re

def extract_code(response: str) -> str:
    fence = '`' * 3
    match = re.search(fence + r'python\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(fence + r'\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()


import pandas as pd

def run_pandas_code(code: str, df) -> str:
    namespace = {'df': df, 'pd': pd}
    try:
        exec(code, namespace)
    except Exception as e:
        return f"Code execution error: {e}"
    result = namespace.get('result', 'No result variable found')
    return str(result)


import re
import ollama
import pandas as pd

def ask_df(df, question: str, model: str = 'llama3.2') -> str:
    schema  = get_df_schema(df)
    prompt  = build_query_prompt(question, schema)
    resp    = ollama.chat(model=model,
                          messages=[{"role": "user", "content": prompt}])
    code    = extract_code(resp["message"]["content"])
    return run_pandas_code(code, df)

## Step 1 — Load Data

In [ ]:
RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
SALES_DF = pd.read_csv(io.StringIO(RETAIL_CSV))
SALES_DF['revenue'] = SALES_DF['price'] * SALES_DF['quantity']

print(f'Shape: {SALES_DF.shape}')
assert SALES_DF.shape == (12, 7)
assert 'revenue' in SALES_DF.columns

## Step 2 — Inspect Schema

In [ ]:
schema = get_df_schema(SALES_DF)
print(schema)

assert '12' in schema
assert 'revenue' in schema
assert 'Widget' in schema or 'Gadget' in schema

## Step 3 — Extract Code (deterministic checks)

In [ ]:
# Verify extract_code with known inputs
_r1 = extract_code('```python\nresult = 42\n```')
assert _r1 == 'result = 42', repr(_r1)
print('extract_code(```python): OK')

_r2 = extract_code('```\nresult = 99\n```')
assert _r2 == 'result = 99', repr(_r2)
print('extract_code(```): OK')

_r3 = extract_code('  result = 0  ')
assert _r3 == 'result = 0', repr(_r3)
print('extract_code(raw): OK')

## Step 4 — Run Pandas Code (deterministic checks)

In [ ]:
_v1 = run_pandas_code('result = df.shape[0]', SALES_DF)
assert _v1 == '12', repr(_v1)
print(f'row count: {_v1}')

_v2 = run_pandas_code("result = df['revenue'].sum()", SALES_DF)
assert '4105' in _v2, repr(_v2)
print(f'total revenue: {_v2}')

_err = run_pandas_code('result = df["bad_col"].sum()', SALES_DF)
assert 'error' in _err.lower()
print(f'error handling: {_err}')

## Step 5 — Full Q&A Pipeline (Ollama)

In [ ]:
a1 = ask_df(SALES_DF, 'What is the total revenue?')
print('Q1:', a1)
assert '4105' in a1

a2 = ask_df(SALES_DF, 'Which product has the highest revenue?')
print('Q2:', a2)
assert 'Gadget' in a2 or 'gadget' in a2.lower()

a3 = ask_df(SALES_DF, 'How many orders are there in each region?')
print('Q3:', a3)
assert isinstance(a3, str) and len(a3.strip()) > 0

a4 = ask_df(SALES_DF, 'What is the average price of all orders?')
print('Q4:', a4)
assert isinstance(a4, str) and len(a4.strip()) > 0

print('\nAll Q&A checks passed.')